In [17]:

from pathlib import Path
import pandas as pd
import numpy as np
import sys

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rest_of_season.config import (
    SEASON,
    CURRENT_WEEK,
    N_SIMULATIONS,
    RANDOM_SEED,
    CURRENT_OUTPUT_DIR
)

TARGET_WEEK = CURRENT_WEEK

WEEKLY_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed" / "weekly"
)

OUTPUT_DIR = CURRENT_OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Season:", SEASON)
print("Projection week:", TARGET_WEEK)
print("Simulations:", N_SIMULATIONS)
print("Project root:", PROJECT_ROOT)
print("ROS output:", OUTPUT_DIR)


Season: 2026
Projection week: 9
Simulations: 50000
Project root: C:\Users\efriedman\Desktop\NFL-Season-Projections
ROS output: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_09


In [18]:

schedule_path = (
    WEEKLY_DATA_DIR / f"{SEASON}_schedule_current.parquet"
)
completed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)
strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)
injury_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

required_files = [
    schedule_path,
    completed_path,
    strength_path,
    injury_path
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required pipeline output missing: {path}"
        )

schedule = pd.read_parquet(schedule_path)
completed = pd.read_parquet(completed_path)
strength = pd.read_parquet(strength_path)
injury_strength = pd.read_parquet(injury_path)

print("Schedule:", schedule.shape)
print("Completed:", completed.shape)
print("Strength:", strength.shape)
print("Injury strength:", injury_strength.shape)


Schedule: (272, 46)
Completed: (121, 46)
Strength: (32, 25)
Injury strength: (32, 30)


In [19]:

# Week 9 rest-of-season input validation
# Read-only: does not modify any saved projections.

assert TARGET_WEEK == 9, (
    f"Expected Week 9, received Week {TARGET_WEEK}"
)

completed_games = schedule.loc[
    (schedule["week"] < TARGET_WEEK)
    & schedule["home_score"].notna()
    & schedule["away_score"].notna()
].copy()

unplayed_prior_games = schedule.loc[
    (schedule["week"] < TARGET_WEEK)
    & (
        schedule["home_score"].isna()
        | schedule["away_score"].isna()
    )
]

future_games = schedule.loc[
    (schedule["week"] >= TARGET_WEEK)
    & schedule["home_score"].isna()
    & schedule["away_score"].isna()
].copy()

print("Snapshot week:", TARGET_WEEK)
print("Completed games:", len(completed_games))
print("Remaining games:", len(future_games))
print("Unplayed games before Week 9:", len(unplayed_prior_games))
print("Total accounted-for games:",
      len(completed_games) + len(future_games))

print("\nCompleted games by week:")
display(
    completed_games.groupby("week")
    .size()
    .to_frame("games")
)

print("\nWeek 8 final scores:")
display(
    completed_games.loc[
        completed_games["week"] == 8,
        ["away_team", "away_score",
         "home_team", "home_score"]
    ].sort_values("home_team")
)

print("\nTeam-strength coverage:")
print("Base strength teams:", strength["team"].nunique())
print("Injury strength teams:",
      injury_strength["team"].nunique())

assert len(unplayed_prior_games) == 0, (
    "Some games before Week 9 are missing final scores."
)

assert len(completed_games) + len(future_games) == 272, (
    "Completed and remaining games do not total 272."
)

assert strength["team"].nunique() == 32
assert injury_strength["team"].nunique() == 32

print("\nWeek 9 input validation passed.")


Snapshot week: 9
Completed games: 121
Remaining games: 151
Unplayed games before Week 9: 0
Total accounted-for games: 272

Completed games by week:


,games
week,
1,16
2,16
3,16
4,16
5,15
6,14
7,14
8,14



Week 8 final scores:


,away_team,away_score,home_team,home_score
110,BAL,10.0,BUF,36.0
119,TEN,36.0,CIN,43.0
108,ARI,33.0,DAL,30.0
113,KC,24.0,DEN,28.0
116,MIN,15.0,DET,19.0
107,CAR,19.0,GB,27.0
112,IND,7.0,JAX,42.0
114,LAC,9.0,LA,24.0
117,NE,24.0,MIA,19.0
115,LV,38.0,NYJ,33.0



Team-strength coverage:
Base strength teams: 32
Injury strength teams: 32

Week 9 input validation passed.


In [20]:

remaining_games = schedule.loc[
    (schedule["week"] >= TARGET_WEEK)
    & schedule["home_score"].isna()
    & schedule["away_score"].isna()
].copy()

remaining_games = remaining_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

assert remaining_games["game_id"].is_unique
assert remaining_games["home_team"].notna().all()
assert remaining_games["away_team"].notna().all()

print("Remaining regular-season games:", len(remaining_games))

display(
    remaining_games[
        ["week", "gameday", "away_team", "home_team"]
    ].head(20)
)


Remaining regular-season games: 151


,week,gameday,away_team,home_team
0,9,2026-11-05,JAX,BAL
1,9,2026-11-08,ARI,SEA
2,9,2026-11-08,CIN,ATL
3,9,2026-11-08,CLE,NO
4,9,2026-11-08,DAL,IND
5,9,2026-11-08,DEN,CAR
6,9,2026-11-08,DET,MIA
7,9,2026-11-08,GB,NE
8,9,2026-11-08,HOU,LAC
9,9,2026-11-08,LA,WAS


In [21]:

completed = completed.loc[
    (completed["week"] < TARGET_WEEK)
    & completed["home_score"].notna()
    & completed["away_score"].notna()
].copy()

home_records = completed[
    ["home_team", "home_score", "away_score"]
].rename(columns={
    "home_team": "team",
    "home_score": "points_for",
    "away_score": "points_against"
})

away_records = completed[
    ["away_team", "away_score", "home_score"]
].rename(columns={
    "away_team": "team",
    "away_score": "points_for",
    "home_score": "points_against"
})

team_results = pd.concat(
    [home_records, away_records],
    ignore_index=True
)

team_results["wins"] = (
    team_results["points_for"] > team_results["points_against"]
).astype(int)

team_results["losses"] = (
    team_results["points_for"] < team_results["points_against"]
).astype(int)

team_results["ties"] = (
    team_results["points_for"] == team_results["points_against"]
).astype(int)

records = team_results.groupby("team", as_index=False).agg(
    wins=("wins", "sum"),
    losses=("losses", "sum"),
    ties=("ties", "sum"),
    games_played=("wins", "size"),
    points_for=("points_for", "sum"),
    points_against=("points_against", "sum")
)

records["point_differential"] = (
    records["points_for"] - records["points_against"]
)

display(records.sort_values(
    ["wins", "point_differential"],
    ascending=[False, False]
))


,team,wins,losses,ties,games_played,points_for,points_against,point_differential
28,SF,6,1,0,7,197.0,139.0,58.0
5,CHI,6,2,0,8,214.0,160.0,54.0
3,BUF,6,1,0,7,233.0,181.0,52.0
20,MIN,6,1,0,7,140.0,93.0,47.0
26,PIT,6,2,0,8,180.0,144.0,36.0
9,DEN,6,2,0,8,172.0,165.0,7.0
14,JAX,5,2,0,7,196.0,111.0,85.0
6,CIN,5,2,0,7,206.0,158.0,48.0
15,KC,5,2,0,7,167.0,131.0,36.0
10,DET,5,2,0,7,203.0,180.0,23.0


In [22]:

required_strength_cols = [
    "team",
    "weekly_team_strength"
]

required_injury_cols = [
    "team",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

for name, frame, columns in [
    ("strength", strength, required_strength_cols),
    ("injury", injury_strength, required_injury_cols)
]:
    missing = set(columns) - set(frame.columns)
    if missing:
        raise KeyError(
            f"{name} missing columns: {sorted(missing)}"
        )

team_snapshot = strength[
    required_strength_cols
].merge(
    injury_strength[required_injury_cols],
    on="team",
    how="left",
    validate="one_to_one"
).merge(
    records,
    on="team",
    how="left",
    validate="one_to_one"
)

assert len(team_snapshot) == 32
assert team_snapshot["team"].is_unique
assert team_snapshot["weekly_team_strength"].notna().all()

print("Teams:", len(team_snapshot))
display(team_snapshot.head(10))


Teams: 32


,team,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength,wins,losses,ties,games_played,points_for,points_against,point_differential
0,BUF,6.296234,-0.252150,6.044084,6,1,0,7,233.0,181.0,52.0
1,JAX,4.547996,-0.450750,4.097246,5,2,0,7,196.0,111.0,85.0
2,LA,4.376166,-0.901650,3.474516,4,4,0,8,194.0,168.0,26.0
3,DEN,4.105679,-0.476400,3.629279,6,2,0,8,172.0,165.0,7.0
4,SF,3.931716,-0.565200,3.366516,6,1,0,7,197.0,139.0,58.0
5,SEA,3.749348,-0.253875,3.495473,4,4,0,8,185.0,159.0,26.0
6,CHI,3.577705,-0.908400,2.669305,6,2,0,8,214.0,160.0,54.0
7,DET,3.225150,-0.915225,2.309925,5,2,0,7,203.0,180.0,23.0
8,MIN,2.861256,-0.365325,2.495931,6,1,0,7,140.0,93.0,47.0
9,PIT,2.844367,-0.615075,2.229292,6,2,0,8,180.0,144.0,36.0


In [23]:

# Validate Week 9 starting standings and team snapshots.
# Read-only; does not overwrite historical files.

standings_check = team_snapshot.copy()

standings_check["record"] = (
    standings_check["wins"].astype(int).astype(str)
    + "-"
    + standings_check["losses"].astype(int).astype(str)
    + "-"
    + standings_check["ties"].astype(int).astype(str)
)

standings_check["remaining_games"] = (
    standings_check["team"].map(
        pd.concat([
            remaining_games["home_team"],
            remaining_games["away_team"]
        ]).value_counts()
    ).fillna(0).astype(int)
)

standings_check["season_games"] = (
    standings_check["games_played"]
    + standings_check["remaining_games"]
)

# Validate all teams and schedules
assert len(standings_check) == 32

assert standings_check["season_games"].eq(17).all(), (
    "At least one team does not have 17 total games."
)

assert (
    standings_check["wins"]
    + standings_check["losses"]
    + standings_check["ties"]
).eq(standings_check["games_played"]).all(), (
    "Wins, losses, and ties do not match games played."
)

assert standings_check[
    "injury_adjusted_team_strength"
].notna().all(), (
    "Missing injury-adjusted team strength."
)

# Sort BEFORE selecting display columns
standings_check = (
    standings_check
    .sort_values(
        ["wins", "ties", "point_differential"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

print("Current NFL standings entering Week 9:")

display(
    standings_check[
        [
            "team",
            "record",
            "games_played",
            "remaining_games",
            "point_differential",
            "weekly_team_strength",
            "injury_adjustment",
            "injury_adjusted_team_strength"
        ]
    ]
)

print("\nAll 32 teams have 17 scheduled games.")
print("Starting standings and team ratings validated.")


Current NFL standings entering Week 9:


,team,record,games_played,remaining_games,point_differential,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength
0,SF,6-1-0,7,10,58.0,3.931716,-0.565200,3.366516
1,CHI,6-2-0,8,9,54.0,3.577705,-0.908400,2.669305
2,BUF,6-1-0,7,10,52.0,6.296234,-0.252150,6.044084
3,MIN,6-1-0,7,10,47.0,2.861256,-0.365325,2.495931
4,PIT,6-2-0,8,9,36.0,2.844367,-0.615075,2.229292
5,DEN,6-2-0,8,9,7.0,4.105679,-0.476400,3.629279
6,JAX,5-2-0,7,10,85.0,4.547996,-0.450750,4.097246
7,CIN,5-2-0,7,10,48.0,1.808836,-0.321975,1.486861
8,KC,5-2-0,7,10,36.0,2.693362,-0.694125,1.999237
9,DET,5-2-0,7,10,23.0,3.225150,-0.915225,2.309925



All 32 teams have 17 scheduled games.
Starting standings and team ratings validated.


In [24]:

team_snapshot["season"] = SEASON
team_snapshot["snapshot_week"] = TARGET_WEEK

remaining_games["season"] = SEASON
remaining_games["snapshot_week"] = TARGET_WEEK

team_snapshot.to_parquet(
    OUTPUT_DIR / "team_snapshot.parquet",
    index=False
)

remaining_games.to_parquet(
    OUTPUT_DIR / "remaining_schedule.parquet",
    index=False
)

team_snapshot.to_csv(
    OUTPUT_DIR / "team_snapshot.csv",
    index=False
)

remaining_games[
    ["week", "gameday", "away_team", "home_team"]
].to_csv(
    OUTPUT_DIR / "remaining_schedule.csv",
    index=False
)

print(f"Saved Week {TARGET_WEEK} ROS inputs.")
print("Output directory:", OUTPUT_DIR)
print("Teams:", len(team_snapshot))
print("Remaining games:", len(remaining_games))


Saved Week 9 ROS inputs.
Output directory: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_09
Teams: 32
Remaining games: 151
